# 05 · Kaggle real-data profiling & evaluation — dataset by dataset, read in place from `/kaggle/input`

> **REAL PUBLIC DATA workflow + one SYNTHETIC third-party dataset.** Nothing is downloaded to your laptop, copied into git, or committed: datasets stay in Kaggle's input storage and are processed *in place*. The output bundle contains **aggregate reports only** (no records, no images, no CSV rows). Every licence on the source cards is **UNVERIFIED**, so no result here may be presented as a publishable real-world claim. Real and synthetic results are never pooled.

**What this notebook establishes on a first run:** *what is actually inside each dataset, and can the CivicConnect data pipeline consume it correctly?* — columns/roles, annotation format, class names, class distribution, missing annotations, duplicate / near-duplicate risk, video-frame relationships, mapping coverage, leakage audits, group-safe holdouts and a `real_holdout` readiness verdict. **No model or provider is called** (`RUN_PROVIDER_EVAL = False`), and nothing is guessed: if a column role, annotation format or class-names file is not established, the run stops for that dataset with the exact setting to supply.

**Settings:** CPU only, **Internet Off** is fine, no GPU. Attach whichever datasets you have; each one runs independently.

| dataset id | attach as | evidence class |
|---|---|---|
| `bmc_mumbai` | Competition data: *Mumbai Nagar Seva / BMC Civic Complaint Resolution 2018-2024* | **third-party SYNTHETIC** (never real-world) |
| `mumbai_nashik_road_surface` (**priority**) | your private Kaggle Dataset made from Mendeley `tj2m7zz4rg` | real, licence reported CC BY 4.0 (unverified here) |
| `rdd2022` (**priority**) | private/trusted Kaggle Dataset of the RDD2022 **India** subset | real, licence unresolved (BY vs BY-SA) |
| `rdd2020` | Kaggle Dataset of the RDD2020 **India** subset | real, reported CC BY-NC 3.0 (unverified) |
| `bharatpothole` | Kaggle dataset `surbhisaswatimohanty/bharatpothole` | real, **licence pending** (publishable evaluation disabled) |

Two passes for the image datasets whose format is not documented (BharatPotHole, Mumbai/Nashik): run once to **profile**, read the report, then fill `IMAGE[...]` below and re-run.

In [ ]:
# --- 0. Locate the CivicConnect repo and a work directory (works on Kaggle and locally) ---------------------
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

def find_repo() -> Path:
    """Return a writable directory that contains ai/inference/schemas.py."""
    marker = Path("ai/inference/schemas.py")
    if os.environ.get("CIVIC_REPO"):
        return Path(os.environ["CIVIC_REPO"])
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / marker).exists():
            return p
    if ON_KAGGLE:                                                   # a Kaggle Dataset containing the repo's `ai/` folder
        for hit in sorted(Path("/kaggle/input").glob("**/ai/inference/schemas.py"))[:1]:
            dst = Path("/kaggle/working/repo")
            if not dst.exists():
                shutil.copytree(hit.parents[2], dst)                # code only (small); /kaggle/input is read-only
            return dst
        if os.environ.get("CIVIC_GIT_URL"):                         # optional (Internet On); use a Kaggle Secret for tokens
            subprocess.run(["git", "clone", "--depth", "1", os.environ["CIVIC_GIT_URL"], "/kaggle/working/repo"], check=True)
            return Path("/kaggle/working/repo")
    raise SystemExit("Cannot find the repo. On Kaggle: add a Dataset containing the repo's `ai/` folder (see ai/training/README.md).")

REPO = find_repo()
sys.path.insert(0, str(REPO))
WORK = Path(os.environ.get("CIVIC_WORKDIR") or ("/kaggle/working" if ON_KAGGLE else tempfile.mkdtemp(prefix="civic_work_")))
WORK.mkdir(parents=True, exist_ok=True)
print(f"repo: {REPO}\nwork: {WORK}\nkaggle: {ON_KAGGLE}")

## 1 · Settings (edit these)
* `ACCEPT_TERMS`: dataset ids whose terms **you** have read on the dataset's own page. Profiling never needs this; *preparing* records does.
* `ACK_UNVERIFIED_LICENSE`: required while a card's licence is still `UNVERIFIED` (always, today). Results then must not be published.
* `PATHS` / `FILES`: only if discovery by name cannot find your mount or training file.
* `ROLE_MAP` (BMC): `{column: role}` after reading `bmc_data_dictionary.csv`; needed only when the report says a role is ambiguous or missing.
* `IMAGE[id]`: `format` (`voc`/`yolo`/`coco`/`folder`), `class_names_file` (YOLO), `group_by` (`dir`/`regex`/`block`) + `group_regex`, `holdout_fraction`, `holdout_seed`. A holdout is only built when a group rule is explicit, so frames of one video can never straddle train and test.

In [ ]:
DATASETS = ["bmc_mumbai", "mumbai_nashik_road_surface", "rdd2022", "rdd2020", "bharatpothole"]   # run any subset
ACCEPT_TERMS = []                      # e.g. ["bmc_mumbai", "rdd2022"] — ONLY after reading each dataset's terms
ACK_UNVERIFIED_LICENSE = False         # set True to prepare records locally while licences are unverified (no publishing)
INPUT_ROOT = None                      # default /kaggle/input (or $CIVIC_KAGGLE_INPUT)
PATHS, FILES = {}, {}                  # e.g. PATHS = {"mumbai_nashik_road_surface": "/kaggle/input/my-road-surface-data"}
ROLE_MAP = {}                          # BMC: {"<column>": "<role>"}
IMAGE = {                              # fill AFTER the profile pass (examples; nothing is assumed)
    # "bharatpothole": {"format": "yolo", "class_names_file": "classes.txt", "group_by": "regex", "group_regex": r"^(.+?)_\d+$", "holdout_fraction": 0.3},
    # "mumbai_nashik_road_surface": {"format": "folder", "group_by": "dir", "holdout_fraction": 0.3},
    # "rdd2022": {"holdout_fraction": 0.25, "block_size": 100},      # RDD is VOC by its documentation; the profile must confirm it
}
RUN_PROVIDER_EVAL = False              # no model/provider call on the first run

# CI hook ONLY: run on an INVENTED /kaggle/input-shaped tree built from the repo's test helpers (NOT real data)
if os.environ.get("CIVIC_KAGGLE_FIXTURE") == "1":
    from ai.training.tests.conftest_real import make_fake_kaggle_input
    INPUT_ROOT = make_fake_kaggle_input(WORK / "fake_kaggle_input", rdd2020=True, bharat=True, nashik=True)
    ACCEPT_TERMS, ACK_UNVERIFIED_LICENSE = list(DATASETS), True
    IMAGE = {"bharatpothole": {"format": "yolo", "class_names_file": "classes.txt", "group_by": "regex", "group_regex": r"^(vid\d+)_", "holdout_fraction": 0.5, "holdout_seed": 2},
             "mumbai_nashik_road_surface": {"format": "folder", "group_by": "dir", "holdout_fraction": 0.5, "holdout_seed": 4},
             "rdd2022": {"block_size": 5, "holdout_fraction": 0.5, "holdout_seed": 1}, "rdd2020": {"block_size": 5, "holdout_fraction": 0.5}}

## 2 · Discovery — what is attached, and exactly what is missing
Read-only: nothing is copied, nothing is written.

In [ ]:
from ai.training.src.data_sources import kaggle_run as kr

CFG = kr.RunConfig(work_dir=WORK, input_root=INPUT_ROOT, accept_terms=ACCEPT_TERMS, ack_unverified_license=ACK_UNVERIFIED_LICENSE, paths=PATHS, files=FILES,
                   role_map=ROLE_MAP, image=IMAGE, run_provider_eval=RUN_PROVIDER_EVAL)
for row in kr.plan(CFG, tuple(DATASETS)):
    print(("FOUND    " if row["found"] else "MISSING  ") + f"{row['dataset_id']:28s} priority={row['priority']:9s} licence={row['licence_status']}  {row['path'] or ''}")
    if not row["found"]:
        print("          ", row["message"])
    if row.get("ambiguity"):
        print("           AMBIGUOUS:", row["ambiguity"])

## 3 · Run, dataset by dataset
Per dataset: **card/provenance → locate → profile → (BMC) column-role validation, leakage audit, synthetic-track evaluation · (images) annotation-format & class-name checks, group-safe holdout, annotation / duplicate / near-duplicate / group audits, `real_holdout` readiness**. A dataset that is absent, needs a setting, or needs your terms acceptance returns a precise status (`NOT_ATTACHED`, `NEEDS_CONFIG`, `NEEDS_TERMS`) and the others still run.

In [ ]:
SUMMARY = kr.run_all(CFG, tuple(DATASETS))
print(f"{'dataset':28s} {'status':14s} {'evidence':22s} {'licence':11s} real_holdout?")
for ds, v in SUMMARY["datasets"].items():
    print(f"{ds:28s} {v['status']:14s} {v['evidence_class']:22s} {v['licence_status']:11s} {v['can_run_real_holdout']}")
print("\n" + SUMMARY["reminder"])

## 4 · Read the reports (what is inside each dataset, and what to do next)

In [ ]:
for ds in DATASETS:
    md = CFG.out / ds / "report.md"
    if md.exists():
        print("=" * 100)
        print(md.read_text(encoding="utf-8")[:6000])

## 5 · Bundle AGGREGATES only (reports, manifests, profiles). Prepared records and images are excluded by construction.

In [ ]:
import hashlib
import zipfile

BUNDLE = kr.bundle(CFG)
names = zipfile.ZipFile(BUNDLE).namelist()
assert not any(n.endswith(("records.jsonl", "pairs.jsonl", "raw.jsonl", ".jpg", ".png", ".csv")) for n in names)
print("bundle (aggregates only):", BUNDLE, "|", BUNDLE.stat().st_size, "bytes | sha256", hashlib.sha256(BUNDLE.read_bytes()).hexdigest()[:16], "|", len(names), "files")
print("REMINDER: BMC is synthetic third-party data; every licence is unverified; do not commit or redistribute data; keep licence status in every report.")